In [ ]:
# base libraries
import numpy as np
import pandas as pd

# plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

# data and preprocessing libraries
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn import preprocessing

import os
from datetime import datetime, timedelta
import math
from scipy import stats
from sklearn.preprocessing import StandardScaler

# Error evaluation libraries
from sklearn.metrics import confusion_matrix

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# Error evaluation libraries
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import mean_absolute_percentage_error as mape
from sklearn.metrics import classification_report

## Process the raw study data
### Note: the crossing decision is manually coded using the study video recordings.

In [3]:
import os
import pandas as pd
import math

# Initialize the model_data DataFrame with the specified columns
model_data = pd.DataFrame(columns=['PID', 'DRate', 'date', 'hr', 'min', 's',
                                   'AGVname', 'StartTime', 'EndTime', 'GazeDuration', 'mean_dist', 'min_dist', 'max_dist', 'std_dist',
                                   'mean_agv_spd', 'min_agv_spd', 'max_agv_spd', 'std_spd', 'task', 'survey'])

# Get the current working directory
current_directory = os.getcwd()

# Specify the relative folder path
folder_path = os.path.join(current_directory, 'textfiles')

# Loop over all files in the folder
for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    df = pd.read_csv(file_path, header=None, names=['line'])

    # Split the filenames
    data_line = filename.split('_')
    del data_line[0]

    # Find the start and end index for each AGV
    # Order should be 2, 1, 3, 4, ..., 16
    agv_order = [2, 1] + list(range(3, 17))
    prefix_list = [f"AGVname=AGV{x}" for x in agv_order]
    start_ind = []
    
    # Find start indices
    for prefix in prefix_list:
        indices = df[df['line'].str.startswith(prefix)].index
        if len(indices) > 0:
            start_ind.append(indices[0] - 2)
        else:
            start_ind.append(None)
            
    # Find end indices
    end_ind = df[df['line'].str.startswith('CheckPoint/Route')].index

    # Extract the data for each AGV
    for i, prefix in enumerate(prefix_list):
        # Skip if the AGV data is missing
        if start_ind[i] is None or (i < len(end_ind) and end_ind[i] is None):
            print(f"Skipping AGV {agv_order[i]}: Missing data")
            continue

        curr_data = data_line.copy()
        curr_data.append(agv_order[i])  # Append AGV number

        gaze_count = 0
        dist_spd_data = pd.DataFrame(columns=['dist', 'spd'])
        
        # Initialize task_result and survey_result to avoid undefined error
        task_result = ""
        survey_result = ""

        for j in range(start_ind[i], end_ind[i]+1 if i < len(end_ind) and end_ind[i] is not None else len(df)):
        
            curr_line = df['line'].iloc[j]
            next_line = df['line'].iloc[j+1] if j+1 < len(df) else ""
        
            if curr_line[0] == 'E':
                fields = curr_line.split()
                if j == start_ind[i]:
                    start_time = fields[1].split('=')[1]
                    curr_data.append(start_time)
                agv_name = fields[0].split('=')[1]
                if agv_name == f'AGV_Sphere{agv_order[i]}':
                    gaze_count += 1
        
            # Check that end_ind[i] exists and is valid
            if curr_line[0] == 'U' and next_line and next_line[0] == 'A':
                curr_fields = curr_line.split()
                if i < len(end_ind) and end_ind[i] is not None and j == end_ind[i] - 2:
                    end_time = curr_fields[3].split('=')[1]
                    curr_data.append(end_time)
        
                user_x = float(curr_fields[0].split('=')[1])
                user_y = float(curr_fields[1].split('=')[1])
                next_fields = next_line.split()
                agv_x = float(next_fields[1].split('=')[1])
                agv_y = float(next_fields[2].split('=')[1])
                agv_spd = float(next_fields[7].split('=')[1])
                d = [math.sqrt((user_x - agv_x)**2 + (user_y - agv_y)**2), agv_spd]
                dist_spd_data.loc[len(dist_spd_data)] = d

            if curr_line[0] == 'C':
                task_result = curr_line.split("Task result is ")[1].split(".")[0]
                survey_result = curr_line.split("Survey result is ")[1].split(".")[0]

        curr_data.extend([
            gaze_count,
            round(dist_spd_data['dist'].mean(), 2) if not dist_spd_data['dist'].empty else None,
            round(dist_spd_data['dist'].min(), 2) if not dist_spd_data['dist'].empty else None,
            round(dist_spd_data['dist'].max(), 2) if not dist_spd_data['dist'].empty else None,
            round(dist_spd_data['dist'].std(), 2) if not dist_spd_data['dist'].empty else None,
            round(dist_spd_data['spd'].mean(), 2) if not dist_spd_data['spd'].empty else None,
            round(dist_spd_data['spd'].min(), 2) if not dist_spd_data['spd'].empty else None,
            round(dist_spd_data['spd'].max(), 2) if not dist_spd_data['spd'].empty else None,
            round(dist_spd_data['spd'].std(), 2) if not dist_spd_data['spd'].empty else None,
            task_result,
            survey_result
        ])

        # Print debugging information
        print(f"Length of curr_data: {len(curr_data)}, Expected: {len(model_data.columns)}")
        print(curr_data)

        if len(curr_data) == len(model_data.columns):
            model_data.loc[len(model_data)] = curr_data
        else:
            print("Error: Length of curr_data does not match number of columns in model_data")

model_data[['Trust', 'Safe', 'Comfort', 'Expect', 'blank']] = model_data['survey'].str.split(' ', expand=True)
model_data.drop(columns=['survey', 'blank'], inplace=True)
model_data.to_csv('Per_Interaction_Data.csv', index=False)

Length of curr_data: 20, Expected: 20
['001', 'High', '2024-5-3', '14', '28', '53', 2, '15:2:48', '15:3:38', 219, 7085.86, 729.06, 12450.38, 4236.37, 8.2, -0.03, 15.22, 6.15, 'true', '10 10 10 9 ']
Length of curr_data: 20, Expected: 20
['001', 'High', '2024-5-3', '14', '28', '53', 1, '15:3:47', '15:4:29', 0, 2296.56, 993.15, 4500.92, 924.77, 5.21, -4.39, 15.19, 3.19, 'true', '10 10 9 10 ']
Length of curr_data: 20, Expected: 20
['001', 'High', '2024-5-3', '14', '28', '53', 3, '15:4:37', '15:5:34', 268, 2950.96, 251.42, 4932.64, 2029.11, 3.24, -5.6, 15.22, 5.28, 'true', '10 9 10 10 ']
Length of curr_data: 20, Expected: 20
['001', 'High', '2024-5-3', '14', '28', '53', 4, '15:5:46', '15:6:37', 779, 5286.88, 546.37, 11770.31, 4347.37, 7.2, -3.5, 15.25, 6.62, 'true', '7 7 7 5 ']
Length of curr_data: 20, Expected: 20
['001', 'High', '2024-5-3', '14', '28', '53', 5, '15:6:41', '15:7:44', 868, 3632.84, 1658.78, 7592.68, 1226.02, 3.31, -3.35, 15.3, 4.72, 'true', '5 6 6 4 ']
Length of curr_data: 

## 1.1 Data Loading

In [ ]:
# Since the datafiles are getting more, is better to ask which one we want
# List all files in the current directory
files = os.listdir()

# Filter out only CSV files
csv_files = [file for file in files if file.endswith('.csv')]
Z
# Print out the list of CSV files
print("Available CSV files:")
for i, file in enumerate(csv_files):
    print(f"{i + 1}. {file}")

# Ask user to choose a file
selection = input("Enter the number of the file you want to read: ")

# Ensure the user input is a valid number
while not selection.isdigit() or int(selection) < 1 or int(selection) > len(csv_files):
    selection = input("Invalid input. Please enter the number of the file you want to read: ")

# Read the selected file
selected_file = csv_files[int(selection) - 1]
data = pd.read_csv(selected_file)

# Getting the dimensions of the dataset
dimensions = data.shape

print(f"Dimensions of {selected_file}: {dimensions}")

In [ ]:
# Display the head of the dataset
data.head()

In [ ]:
# Check for NaN values in the DataFrame
nan_counts = data.isna().sum()

# Display columns with NaN counts greater than 0
nan_columns = nan_counts[nan_counts > 0]
print("Columns with NaN values:")
print(nan_columns)

In [ ]:
# Get the list of columns in the dataset
columns_list = data.columns.tolist()

# Print the list of columns
print(columns_list)

### Visualizing the user and AGV location in the manufacturing plant 
#### This visualization helps us see the user's behavior towards the presence of the AGV and whether, towards a particular AGV, users exhibit normal behaviors or not, and why.

#### Use the different shape to represent AGVs. They're the same as the user, which are confusing.

In [ ]:
# Get unique AGV names
agv_names = data['AGVname'].unique()

# Loop through each AGV
for agv_name in agv_names:
    # Filter data for the current AGV
    filtered_data = data[(data['AGVname'] == agv_name) & (data['DRate'] == 'High') & (data['PID'] == 12)]

    # Create a scatter plot with a warm coloring gradient
    plt.figure(figsize=(7, 6))

    # Get the number of data points
    num_points = len(filtered_data)

    # Create a color gradient based on the order of data points
    colors = np.arange(num_points) / num_points

    # Plot AGV data
    plt.scatter(filtered_data['AGV_X'], filtered_data['AGV_Y'], marker='s', c=colors, cmap='RdYlGn', alpha=0.9, label='AGV Data')

    # Plot User data on the same plot
    plt.scatter(filtered_data['User_X'], filtered_data['User_Y'], c=colors, cmap='RdYlGn', alpha=0.99, label='User Data')

    # Set x-axis and y-axis limits
    plt.xlim(0, 20000)
    plt.ylim(0, 20000)

    # Set labels and title
    plt.xlabel('X')
    plt.ylabel('Y')
    plt.title(f'Scatter Plot of AGV_X, AGV_Y, User_X, and User_Y for {agv_name}')

    # Add legend
    plt.legend()

    # Show the plot
    plt.colorbar(label='Data Point Order')  # Add a colorbar to indicate the order of data points

    plt.grid()
    
    # Generate a timestamp
    timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

    # Define the filename with timestamp
    filename = f"Scatter_Plot_of_the_Expected_X_Expected_Y_{agv_name}_{timestamp}.png"
    
    plt.savefig(save_dir + filename, dpi = 300)
    
    plt.show()

#### These are box plots that represent the distribution of user_x and user_y in their interaction with different AGVs.

In [ ]:
# Assuming 'AGV_distance' is the column you want to find the minimum value for
unique_pairs = data.groupby(['AGVname', 'SCN']).size().reset_index(name='count')

In [ ]:
# Create subplots for User_X and User_Y box plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
boxprops = dict(linewidth=2, color='blue')
flierprops = dict(markerfacecolor='red', markersize=8, marker='s')

# Create a box plot for User_X
bx1 = ax1.boxplot([data[(data['AGVname'] == agv_name) & (data['SCN'] == scn)]['User_X'] for agv_name, scn in zip(unique_pairs['AGVname'], unique_pairs['SCN'])],
                  labels=[f'{agv_name} - {scn}' for agv_name, scn in zip(unique_pairs['AGVname'], unique_pairs['SCN'])],
                  boxprops=boxprops,
                  flierprops=flierprops)

ax1.set_title('Box Plot of User_X for Each AGVname - SCN')
ax1.set_xlabel('AGVname - SCN')
ax1.set_ylabel('User_X')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=90, ha='right')  # Rotate x-axis labels

# Create a box plot for User_Y
bx2 = ax2.boxplot([data[(data['AGVname'] == agv_name) & (data['SCN'] == scn)]['User_Y'] for agv_name, scn in zip(unique_pairs['AGVname'], unique_pairs['SCN'])],
                  labels=[f'{agv_name} - {scn}' for agv_name, scn in zip(unique_pairs['AGVname'], unique_pairs['SCN'])],
                  boxprops=boxprops,
                  flierprops=flierprops)

ax2.set_title('Box Plot of User_Y for Each AGVname - SCN')
ax2.set_xlabel('AGVname - SCN')
ax2.set_ylabel('User_Y')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=90, ha='right')  # Rotate x-axis labels

# Define the filename using f-string or other appropriate method
filename = f"box_plots_user_x_y_{timestamp}.png"

# Save the plots in the specified directory
plt.savefig(save_dir + filename, dpi = 300)

# Show the plots
plt.show()

# 2. Adding Trust as a Ground Truth Value

In [ ]:
data['Trust'] = ""

In [ ]:
data.head()

In [ ]:
model_data  = pd.read_csv('model_data.csv')

In [ ]:
# Assuming you want to rename the 'OldColumnName' to 'NewColumnName'
model_data.rename(columns={'AGV_name': 'AGVname'}, inplace=True)

In [ ]:
# Extract numeric part from 'AGVname' and create a new column 'AGV_number'
data['AGVname'] = data['AGVname'].str.extract('(\d+)')

# Convert 'AGVname' column to integer type
data['AGVname'] = data['AGVname'].astype(int)

column_type = data['AGVname'].dtype

print(f"The type of values in the 'AGVname' column is: {column_type}")

In [ ]:
# Display the updated DataFrame
data.iloc[3050:3061]

In [ ]:
# Create a mapping dictionary from model_data
trust_dict = model_data.set_index(['AGVname', 'SCN', 'PID'])['Trust'].to_dict()

# Map the Trust values from model_data to data_by_sec based on the unique pairs ('AGVname', 'SCN', 'PID')
data['Trust'] = data.set_index(['AGVname', 'SCN', 'PID']).index.map(trust_dict)

In [ ]:
data.shape

# 3. Data Analysis

### Datapoints within the 25th-75th percentile of the user's y-axis location

#### We want to focus on the datapoints related to the actual interaction between the human and AGV as much as possible. 
#### Obviously, datapoints before the user notices the AGV cannot help us model the user's behavior towards the AGV or measure trust.
#### As a result, want to keep the 25th-75th percentiles for each unique pair of AGVname-SCN.

In [ ]:
data['User_Y'].describe()

#### Capture as many data points focusing on the interaction of AGV and user as possible.

In [ ]:
# Define a function to filter each group based on percentiles
def filter_by_percentiles(group):
    percentile_25 = group['User_Y'].quantile(0.0)
    percentile_75 = group['User_Y'].quantile(1)
    return group[(group['User_Y'] >= percentile_25) & (group['User_Y'] <= percentile_75)]

# Apply the filtering function to each group defined by 'AGVname' and 'SCN'
filtered_data = data.groupby(['AGVname', 'SCN'], group_keys=False).apply(filter_by_percentiles)


# Print the filtered DataFrame
filtered_data.iloc[0:10]

### User Pitch, Yaw, Roll Data

#### We wanted to understand what these data represent. Gaining a sense of this data helps us justify different machine learning algorithms.

In [ ]:
'''
# Replace 'YourColumnName' with the actual column names in your DataFrame
time_column = 'Timestamp'
user_pitch_column = 'User_Pitch'
user_yaw_column = 'User_Yaw'
user_roll_column = 'User_Roll'
agv_pitch_column = 'AGV_Pitch'
agv_yaw_column = 'AGV_Yaw'
agv_roll_column = 'AGV_Roll'
agv_name_column = 'AGVname'
scn_column = 'SCN'

# Get unique combinations of AGVname and SCN
unique_combinations = filtered_data_by_sec[[agv_name_column, scn_column]].drop_duplicates()

# Plot data for each unique combination
for index, row in unique_combinations.iterrows():
    agv_name = row[agv_name_column]
    scn_value = row[scn_column]

    plt.figure(figsize=(15, 8))

    # Plot User's data
    user_data = filtered_data_by_sec[(filtered_data_by_sec[agv_name_column] == agv_name) & (filtered_data_by_sec[scn_column] == scn_value)]
    plt.plot(user_data[time_column], user_data[user_pitch_column], label='User Pitch')
    plt.plot(user_data[time_column], user_data[user_yaw_column], label='User Yaw')
    plt.plot(user_data[time_column], user_data[user_roll_column], label='User Roll')

    # Plot AGV's data
    plt.plot(user_data[time_column], user_data[agv_pitch_column], label=f'{agv_name} Pitch')
    plt.plot(user_data[time_column], user_data[agv_yaw_column], label=f'{agv_name} Yaw')
    plt.plot(user_data[time_column], user_data[agv_roll_column], label=f'{agv_name} Roll')

    plt.legend()
    plt.xlabel('Time')
    plt.ylabel('Angle')
    plt.title(f'User and AGV Angles Over Time - {agv_name}, SCN {scn_value}')
    plt.show()
'''

### Scatter plot for each specific AGV 
#### The input is the AGV's name, and the output showcases 26 scatter plots, each representing a specific user location towards that AGV with its corresponding behavior.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# with columns User_X, User_Y, AGV_X, AGV_Y, AGVname, SCN, and PID

# Filter data for AGVname
agv_data = [filtered_data['AGVname'] == 'AGV1']

# Separate SCNs (NSL and SLD)
nsl_data = agv_data[agv_data['SCN'] == 'NSL']
sld_data = agv_data[agv_data['SCN'] == 'SLD']

# Get unique PIDs for NSL and SLD
unique_nsl_pids = nsl_data['PID'].unique()
unique_sld_pids = sld_data['PID'].unique()

# Create subplots for each unique PID with NSL and SLD side by side
fig, axes = plt.subplots(nrows=len(unique_nsl_pids), ncols=2, figsize=(12, 6 * len(unique_nsl_pids)))

# Iterate over unique PIDs
for i, pid in enumerate(unique_nsl_pids):
    # Filter data for the current PID in NSL
    nsl_pid_data = nsl_data[nsl_data['PID'] == pid]
    # Filter data for the current PID in SLD
    sld_pid_data = sld_data[sld_data['PID'] == pid]

    # Plot User and AGV points for NSL
    axes[i, 0].scatter(nsl_pid_data['User_X'], nsl_pid_data['User_Y'], label=f'NSL - PID: {pid} (U)', alpha=0.9)
    axes[i, 0].scatter(nsl_pid_data['AGV_X'], nsl_pid_data['AGV_Y'], label=f'NSL - PID: {pid} (AGV)', alpha=0.9)
    axes[i, 0].set_title(f'Scatter Plot for NSL - PID: {pid}')
    axes[i, 0].legend()
    axes[i, 0].set_ylim(0, 12000)  # Set y-axis limits

    # Plot User and AGV points for SLD
    axes[i, 1].scatter(sld_pid_data['User_X'], sld_pid_data['User_Y'], label=f'SLD - PID: {pid} (U)', alpha=0.9)
    axes[i, 1].scatter(sld_pid_data['AGV_X'], sld_pid_data['AGV_Y'], label=f'SLD - PID: {pid} (AGV)', alpha=0.9)
    axes[i, 1].set_title(f'Scatter Plot for SLD - PID: {pid}')
    axes[i, 1].legend()
    axes[i, 1].set_ylim(0, 12000)  # Set y-axis limits

# Adjust layout
plt.tight_layout()

# Define the filename using f-string or other appropriate method
filename = f"data_for_AGVname_{timestamp}.png"

# Save the plots in the specified directory
plt.savefig(save_dir + filename, dpi = 300)

# Show the plots
plt.show()

### Make a copy of data

In [ ]:
data_with_distance = filtered_data.copy()

data_with_distance.head()

### Dropping Timestamp Column
#### When working with machine learning algorithms, we mostly need numerical data; that's why we'd like to change the format of the timestamp to the actual second of the day the particular interaction happened.

In [ ]:
# Convert the 'Timestamp' column to pandas datetime object
data_with_distance['Timestamp'] = pd.to_datetime(data_with_distance['Timestamp'])

# Convert the 'Timestamp' column to seconds
data_with_distance['time_seconds'] = data_with_distance['Timestamp'].dt.hour * 3600 + data_with_distance['Timestamp'].dt.minute * 60 + data_with_distance['Timestamp'].dt.second

data_with_distance.head()

In [ ]:
data_with_distance.shape

In [ ]:
data_with_distance = data_with_distance.drop(columns=['Timestamp'])

### Difference Columns

#### We define some columns as difference columns, as the difference can convey more meaning, indicating whether the AGV or the user moved or stayed, than the absolute value.

In [ ]:
# Define the columns for which differences need to be calculated
difference_columns = ['User_X', 'User_Y', 'User_Z', 'AGV_X', 'AGV_Y', 'AGV_Z', 'time_seconds']

# Group by unique pairs of (AGVname, PID, SCN)
grouped_data = data_with_distance.groupby(['AGVname', 'PID', 'SCN'])

# Initialize an empty list to store individual group dataframes
result_dfs = []

# Iterate over groups and calculate differences
for group_name, group_df in grouped_data:
    # Calculate differences for the specified columns
    differences = group_df[difference_columns].diff().rename(lambda x: f'{x}_difference', axis=1)
    
    # Concatenate differences with the original dataframe
    result_df = pd.concat([group_df, differences], axis=1)
    
    # Append the result to the list
    result_dfs.append(result_df)

# Concatenate all individual dataframes into the final result
data_with_distance_difference = pd.concat(result_dfs, ignore_index=True)

# Drop NaN rows resulting from differences
data_with_distance_difference = data_with_distance_difference.dropna()

# Display the final result
data_with_distance_difference.head()

In [ ]:
'''
desired_agvname = 'AGV1'
desired_pid = 3
desired_scn = 'NSL'

# Filter the final_result DataFrame based on the specified values
data_with_distance_difference = data_with_distance_difference[(data_with_distance_difference['AGVname'] == desired_agvname) & 
                                (data_with_distance_difference['PID'] == desired_pid) & 
                                (data_with_distance_difference['SCN'] == desired_scn)]

# Display the filtered result
data_with_distance_difference.head()
'''

### Visualizing User_Coordinates_Difference_Dataframe

#### For each specific user, AGV, and its behavior, we can observe the differences in y-coordinates. 
#### This gives us a general idea of the distance threshold.

In [ ]:
'''
# Specify the AGVname you want to plot
specific_agvname = 'AGV1'
specific_PID = 8
specific_SCN = 'SLD'

# Filter data for the specific AGVname, PID, and SCN
filtered_data = data_with_distance_difference[
    (data_with_distance_difference['AGVname'] == specific_agvname) &
    (data_with_distance_difference['PID'] == specific_PID) &
    (data_with_distance_difference['SCN'] == specific_SCN)
]

# Create a line plot for User_Y_difference with row number on the x-axis
plt.figure(figsize=(14, 8))
sns.lineplot(x=filtered_data.index, y='User_Y_difference', hue='SCN', style='AGVname', data=filtered_data)
plt.title('User_Y_difference over Row Number, Grouped by SCN and AGVname')
plt.xlabel('Row Number')
plt.ylabel('User_Y_difference')
plt.show()
'''

In [ ]:
# Assuming data_with_distance_difference is your DataFrame
data_with_distance_difference['User_distance'] = np.sqrt(
    data_with_distance_difference['User_X_difference']**2 +
    data_with_distance_difference['User_Y_difference']**2
)

data_with_distance_difference['AGV_distance'] = np.sqrt(
    ((data_with_distance_difference['User_X'] - data_with_distance_difference['AGV_X']) ** 2) +
    ((data_with_distance_difference['User_Y'] - data_with_distance_difference['AGV_Y']) ** 2) +
    ((data_with_distance_difference['User_Z'] - data_with_distance_difference['AGV_Z']) ** 2)
) 
data_with_distance_difference.loc[0:20]

### Visualize User_distance, based on each specific pair of (AGVname, SCN)

In [ ]:
# Assuming data_with_distance_difference is your DataFrame
'''
unique_pairs = data_with_distance_difference.groupby(['AGVname', 'SCN']).size().reset_index(name='count')

# Iterate over unique pairs and plot for each AGVname and SCN
for i, (agv_name, scn) in enumerate(zip(unique_pairs['AGVname'], unique_pairs['SCN'])):
    # Filter data for the current AGVname and SCN
    agv_data = data_with_distance_difference[
        (data_with_distance_difference['AGVname'] == agv_name) & 
        (data_with_distance_difference['SCN'] == scn)
    ]
    
    # Create a line plot for User_distance with counter (row number) on the x-axis
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=range(1, len(agv_data) + 1), y='User_distance', data=agv_data)
    plt.title(f'User_distance over Counter for {agv_name} - {scn}')
    plt.xlabel('Counter')
    plt.ylabel('User_distance')
    plt.show()
'''

### Visualizing User_distance

In [ ]:
# Assuming data_with_distance_difference is your DataFrame
plt.figure(figsize=(10, 6))
sns.lineplot(x=range(1, len(data_with_distance_difference) + 1), y='User_distance', data=data_with_distance_difference)
plt.title('User_distance')
plt.xlabel('Counter')
plt.ylabel('User_distance')

# Define the filename using f-string or other appropriate method
filename = f"users distance_{timestamp}.png"

# Save the plots in the specified directory
plt.savefig(save_dir + filename, dpi = 300)

plt.show()

In [ ]:
data_with_distance_difference['User_distance'].describe()

### Different Percentiles 
#### We were looking for significant jumps, but that wasn't the case. We can see that up to 30% of the user-distance variables are zero, and 70% are non-zero, which makes sense. This indicates that our hardware accurately distinguishes between movement and stopping.

In [ ]:
percentiles = [0.1, 0.25, 0.3, 0.4, 0.5, 0.6, 0.75, 0.8, 0.9, 1]

# Calculate specific percentiles of User_distance
user_distance_percentiles = data_with_distance_difference['User_distance'].quantile(percentiles)

# Print the results
for percentile, value in zip(percentiles, user_distance_percentiles):
    print(f'{percentile * 100}% Quantile: {value}')

In [ ]:
percentiles_ = [0.28, 0.29, 0.3, 0.31, 0.32, 0.33, 0.34, 0.35, 0.36, 0.37, 0.38, 0.39, 0.40, 0.41, 0.415, 0.42, 0.43, 0.44, 0.45, 0.46, 
                0.47, 0.48, 0.49, 0.50, 0.51, 0.52, 0.53, 0.54, 0.55, 0.56, 0.57, 0.58]

# Calculate specific percentiles of User_distance
user_distance_percentiles = data_with_distance_difference['User_distance'].quantile(percentiles_)

# Print the results
for percentile, value in zip(percentiles_, user_distance_percentiles):
    print(f'{percentile * 100}% Quantile: {value}')

In [ ]:
!pip install kneed
from kneed import KneeLocator

# Find the elbow point using the KneeLocator
kneedle = KneeLocator(percentiles_, user_distance_percentiles, curve='convex', direction='increasing')
elbow_point = kneedle.knee

#
print(elbow_point)

# Create a line plot for percentiles
plt.figure(figsize=(10, 6))
sns.lineplot(x=percentiles_, y=user_distance_percentiles, marker='o')
plt.title('User_distance Percentiles')
plt.xlabel('Percentiles')
plt.ylabel('User_distance')
plt.vlines(elbow_point, 0, 50, linestyles ="dashed", colors ="k")

filename = f'Treshhold Determination_{timestamp}.png'

plt.savefig(save_dir + filename, dpi = 300)

plt.show()

#### Replacing the AGV_number with AGVname so that we can work with numerical values if needed, such as when building the correlation matrix.

In [ ]:
# Display the updated DataFrame
data_with_distance_difference.iloc[1400000:1400010]

### The minimum index for AGV_distance

#### For each unique pair of AGV_number and SCN, we want to know the index of the minimum value of AGV_distance. 

In [ ]:
# Assuming 'AGV_distance' is the column you want to find the minimum value for
unique_pairs = data_with_distance_difference.groupby(['AGVname', 'SCN'])

min_distances = unique_pairs['AGV_distance'].min()

# Get the index of the minimum value within each group
idx_min_distances = unique_pairs['AGV_distance'].idxmin()

# Create a DataFrame with 'AGV_number', 'SCN', 'Min AGV_distance', and 'Index of Min Distance'
min_idx_agv_distance = pd.DataFrame({
    'AGVname': idx_min_distances.index.get_level_values('AGVname'),
    'SCN': idx_min_distances.index.get_level_values('SCN'),
    'Min AGV_distance': min_distances.values,
    'Index of Min Distance': idx_min_distances
})

# Reset the index to remove the current index (AGV_number and SCN become regular columns)
min_idx_agv_distance = min_idx_agv_distance.reset_index(drop=True)

min_idx_agv_distance_sorted = min_idx_agv_distance.sort_values(by='AGVname', key=lambda x: x.astype(int))

# Print the sorted DataFrame
min_idx_agv_distance_sorted

In [ ]:
df = min_idx_agv_distance_sorted.drop('Min AGV_distance', axis=1)

# Create a figure and axis
fig, ax = plt.subplots(figsize=(4, 6))

# Hide the axes
ax.axis('off')

# Plot the DataFrame as a table
table = ax.table(cellText=df.values, colLabels=df.columns, cellLoc = 'center', loc='center', colColours=['#f0f0f0']*len(df.columns))

# Save the figure as an image
filename = f'min_idx_agv_distance_sorted_table_{timestamp}.png'

plt.savefig(save_dir + filename, dpi = 300)

# Show the plot
plt.show()

In [ ]:
# Get 'AGV_number' from the reset DataFrame
x_values = min_idx_agv_distance_sorted['AGVname']

# Assuming 'Min AGV_distance' is the column representing y-axis
y_values = min_idx_agv_distance_sorted['Min AGV_distance']

# Get 'SCN' values
scn_values = min_idx_agv_distance_sorted['SCN']

# Define colors based on 'SCN' values
colors = ['blue' if scn == 'SLD' else 'red' for scn in scn_values]

# Create a bigger plot
plt.figure(figsize=(10, 6))

# Plotting the stem plot with conditional colors for 'SLD'
plt.stem(x_values[scn_values == 'SLD'], y_values[scn_values == 'SLD'], linefmt='-b', markerfmt='bo', basefmt='-g', bottom=0)

# Plotting the stem plot with conditional colors for 'NSL'
plt.stem(x_values[scn_values == 'NSL'], y_values[scn_values == 'NSL'], linefmt='-r', markerfmt='ro', basefmt='-g', bottom=0)

plt.xlabel('AGV_number')
plt.ylabel('Min AGV_distance')
plt.title('Stem Plot of Min AGV_distance for Each Group')

# Save the plot to a file
filename = f'Stem_Plot_Min_AGV_distance_{timestamp}.png'

plt.savefig(save_dir + filename, dpi = 300)

# Show the plot
plt.show()

In [ ]:
# Assuming data_with_distance_difference is your DataFrame
# Group by 'AGV_number' and 'SCN', then find the row with the minimum 'AGV_spd'
min_agv_spd_rows = data_with_distance_difference.groupby(['AGVname', 'SCN'])['AGV_spd'].idxmin()

# Extract the corresponding rows based on the index
result_df = data_with_distance_difference.loc[min_agv_spd_rows, ['AGV_distance', 'AGV_spd']]

# Print or inspect the result
result_df

In [ ]:
# List of row indexes
row_indexes = [205, 585, 1749, 2013, 2906, 2739, 4182, 3433, 5034, 5291, 5587, 6160, 6759, 7237, 7930, 8361, 9008, 8708, 10698, 10308, 11185, 11148, 12121, 12442, 13104, 13337, 14952, 14526, 15550, 15537, 16816, 16920]

# Extract 'AGV_number', 'SCN', and 'AGV_distance' for the provided row indexes
selected_rows = data_with_distance_difference.loc[row_indexes, ['AGVname', 'SCN', 'AGV_distance']]

# Save selected_rows as a CSV file
selected_rows.to_csv('selected_rows.csv', index=False)

# Create a table-like plot using Matplotlib
fig, ax = plt.subplots(figsize=(6, 8))
ax.axis('off')  # Turn off axis labels

table_data = [selected_rows.columns] + selected_rows.values.tolist()
ax.table(cellText=table_data, colLabels=None, cellLoc='center', loc='center')

# Save the plot to a file
filename = f'selected_rows_table_{timestamp}.png'

# Save the plot as a PNG file
plt.savefig(save_dir + filename, dpi = 300, bbox_inches='tight')

# Show the plot if needed
plt.show()

### data_with_threshold
#### Setting the threshold at 0, turning it into a dataset

In [ ]:
# Add a new 'behavior' column based on User_distance values
data_with_threshold = data_with_distance_difference.copy()

In [ ]:
data_with_threshold['Behavior'] = ['Stop' if distance == 0 else 'Moving' for distance in data_with_threshold['User_distance']]

# Display the updated DataFrame
data_with_threshold.head()

In [ ]:
#'SCN' is the column where replacement is needed
data_with_threshold['SCN'] = data_with_threshold['SCN'].replace('NSL', 0).replace('SLD', 1)

#'Behavior' is the column where replacement is needed
data_with_threshold['Behavior'] = data_with_threshold['Behavior'].replace('Stop', 0).replace('Moving', 1)

In [ ]:
# Setting up the figure
plt.figure(figsize=(12, 11))

# Check correlation matrix
correlation_matrix = data_with_threshold.corr()

#Draw the heat map
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', fmt=".2f", linewidths=0.5)

plt.title('Correlation Heatmap')

plt.show

## Importing Libraries

In [ ]:
from scipy.spatial.distance import euclidean
from typing import List, Tuple

### Run below if you want to test the code on a smal portion of the dataset

In [ ]:
'''
# Testing the frechet distance on a small proportion of the threshold data

# Choose a small proportion of the dataset (e.g. 10%)
proportion = 0.1 
trajectory_deviation_data_set = data_with_threshold.sample(frac=proportion, random_state=42).copy()
'''

In [ ]:
# Using threshold data
trajectory_deviation_data_set = data_with_threshold.copy()

In [ ]:
# Sorting the Data
trajectory_deviation_data_set.sort_values(by=['AGVname', 'SCN', 'PID'],inplace=True)

In [ ]:
# Asking about the number of seconds horizon
n_points = int(input("Enter the number of seconds for horizon to calculate the frechet distance: "))

# Function to generate expected trajectory points from the current position to the end
def generate_expected_trajectory(start: Tuple[float, float], end: Tuple[float, float]) -> List[Tuple[float, float]]:
    return [(start[0] + (end[0] - start[0]) * i / (n_points - 1), start[1] + (end[1] - start[1]) * i / (n_points - 1)) for i in range(n_points)]

# Function to select n points from the actual path
def select_actual_path_points(current_position: Tuple[float, float], path: List[Tuple[float, float]]) -> List[Tuple[float, float]]:
    distances = [euclidean(current_position, p) for p in path]
    current_index = distances.index(min(distances))
    return path[current_index:current_index + n_points]

# Fréchet Distance calculation
def frechet_distance(p: List[Tuple[float, float]], q: List[Tuple[float, float]]) -> float:
    ca = np.full((len(p), len(q)), -1.0, dtype=float)
    return _c(ca, p, q, len(p)-1, len(q)-1)

def _c(ca, p, q, i, j):
    if ca[i, j] > -1:
        return ca[i, j]
    elif i == 0 and j == 0:
        ca[i, j] = euclidean(p[0], q[0])
    elif i > 0 and j == 0:
        ca[i, j] = max(_c(ca, p, q, i-1, 0), euclidean(p[i], q[0]))
    elif i == 0 and j > 0:
        ca[i, j] = max(_c(ca, p, q, 0, j-1), euclidean(p[0], q[j]))
    elif i > 0 and j > 0:
        ca[i, j] = max(min(_c(ca, p, q, i-1, j), _c(ca, p, q, i-1, j-1), _c(ca, p, q, i, j-1)), euclidean(p[i], q[j]))
    else:
        ca[i, j] = float('inf')
    return ca[i, j]

# Processing each interaction group
def process_interaction_group(group):
    actual_path = list(zip(group['User_X'], group['User_Y']))
    end_point = actual_path[-1]  # endpoint to the last coordinate of the group
    frechet_distances = []

    for current_position in actual_path:
        expected_trajectory = generate_expected_trajectory(current_position, end_point)
        selected_actual_path = select_actual_path_points(current_position, actual_path)
        frechet_dist = frechet_distance(expected_trajectory, selected_actual_path)
        frechet_distances.append(frechet_dist)

    return frechet_distances

# MAB: I added this for the upcoming "for" loop, please check to see whther it makes sense
grouped = trajectory_deviation_data_set.groupby(['PID', 'AGVname', 'Behavior'])

# Adding a sequential identifier within each group in the original dataset
trajectory_deviation_data_set['Point_ID'] = trajectory_deviation_data_set.groupby(['PID', 'AGVname', 'Behavior']).cumcount()

# Initializing an empty DataFrame for the results
fretchet_df = pd.DataFrame(columns=['PID', 'AGVname', 'Behavior', 'Point_ID', 'Frechet_Distance'])

# Processing each group and appending the results to the DataFrame
for (pid, agv_number, behavior), group in grouped:
    frechet_distances = process_interaction_group(group)
    
    for point_id, distance in enumerate(frechet_distances):
        new_row_df = pd.DataFrame({
            'PID': [pid],
            'AGVname': [agv_number],
            'Behavior': [behavior],
            'Point_ID': [point_id],  # Include the sequential identifier
            'Frechet_Distance': [distance]  # Each distance in its own cell
        })
        fretchet_df = pd.concat([fretchet_df, new_row_df], ignore_index=True)

# Saving the results as a CSV file
# fretchet_df.to_csv('expanded_results_dataframe.csv', index=False)

# Merging the fretchet_df DataFrame with the original dataset using the keys
trajectory_deviation_data_set_merged = pd.merge(trajectory_deviation_data_set, fretchet_df, on=['PID', 'AGVname', 'Behavior', 'Point_ID'], how='left')

# Saving the merged DataFrame to a new CSV file
# merged_df.to_csv('merged_results_dataframe.csv', index=False)

trajectory_deviation_data_set_merged.head(40)

# 5. Feature Selection

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

normalized_data = scaler.fit_transform(data)

data_normalized = pd.DataFrame(normalized_data, columns=data.columns)

data_normalized.head

In [ ]:
# Get the summary statistics of the "Trust" column
trust_summary = data_normalized['Trust'].describe()

# Print the summary statistics
print(trust_summary)

In [ ]:
# Select columns for plotting and normalization
columns_to_plot = ['User_X', 'User_Y', 'GazeOrigin_X', 'GazeOrigin_Y', 'User_Roll', 'User_Pitch', 'User_Yaw']
row_interval = 100000

scaler = MinMaxScaler()
data_normalized = pd.DataFrame(scaler.fit_transform(data[columns_to_plot]), columns=columns_to_plot)

# Create a scatter plot for each selected column with the specified interval
plt.figure(figsize=(10, 10))  # Adjust the figure size as needed

for idx, column in enumerate(columns_to_plot):
    plt.scatter(data_normalized[column][::row_interval], [column] * len(data_normalized[column][::row_interval]), label=column, s=100, alpha=0.9)

plt.xlabel('Normalized Values')
plt.title('Scatter Plot of Normalized Column Values')
plt.legend()
plt.grid(True)
plt.ylim(-1, len(columns_to_plot))  # Set y-axis limits from -1 to number of columns
plt.xlim(-0.05, 1.05)  # Set x-axis limits from -0.05 to 1.05
# Move the legend outside of the plot
plt.legend(loc='upper left', bbox_to_anchor=(1, 0.5))

# Save the figure as an image
filename = 'Collected_Data.png'

plt.savefig(save_dir + filename, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Select columns for plotting and normalization
columns_to_plot = ['User_X', 'User_Y', 'GazeOrigin_X', 'GazeOrigin_Y', 'User_Roll', 'User_Pitch', 'User_Yaw', 'AGV_distance', 'Behavior', 'Trust']

# Combine 'Trust' with each column for pairplot
pairplot_data = data_normalized[columns_to_plot]

# Plot pairplot
plt.figure(figsize=(10, 8))
sns.pairplot(pairplot_data, kind='scatter', diag_kind='kde', plot_kws={'alpha': 0.6})

plt.suptitle('Pairplot with KDE for Selected Columns', y=1.02)
plt.tight_layout()

# Save the figure as an image
filename = 'Pairplot with KDE for Selected Columns.png'
plt.savefig(save_dir + filename, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Select columns for joint kernel density estimate
columns_for_joint_kde = ['User_X', 'User_Y', 'User_Roll', 'User_Pitch', 'User_Yaw', 'Trust']

# Create joint kernel density estimate plot for selected columns
plt.figure(figsize=(10, 8))
sns.pairplot(data_normalized[columns_for_joint_kde], kind='kde', diag_kind='kde', plot_kws={'alpha': 0.6})

plt.suptitle('Joint Kernel Density Estimate for Selected Columns', y=1.02)
plt.tight_layout()

# Save the figure as an image
filename = 'Joint Kernel Density Estimate for Selected Columns.png'
plt.savefig(save_dir + filename, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
# Select columns for plotting and normalization
columns_to_plot = ['User_X', 'User_Y','User_Roll', 'User_Pitch','User_Yaw', 'Trust']

# Plot combined KDE plot for selected columns
plt.figure(figsize=(10, 6))

# Loop through each column and plot its KDE
for column in columns_to_plot:
    sns.kdeplot(data_normalized[column], label=column, shade=True)

plt.xlabel('Value')
plt.ylabel('Density')
plt.title('Kernel Density Estimation (KDE) for Selected Columns')
plt.legend()
plt.grid(True)

# Save the figure as an image
filename = 'Kernel Density Estimation (KDE) for Selected Columns - 2.png'

plt.savefig(save_dir + filename, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Select columns for plotting and normalization
columns_to_plot = ['User_X', 'User_Y', 'GazeOrigin_X', 'GazeOrigin_Y', 'User_Roll', 'User_Pitch', 'User_Yaw', 'AGV_distance', 'Behavior', 'Trust']

# Create a violin plot for selected columns
plt.figure(figsize=(10, 6))
ax = sns.violinplot(data=data_normalized[columns_to_plot], inner='quartile', palette='muted')
plt.xlabel('Columns')
plt.ylabel('Value')
plt.title('Violin Plot of Selected Columns')
plt.grid(True)

# Rotate x-axis labels by 45 degrees
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)

# Save the figure as an image
filename = 'Violin_Plot_of_Selected_Columns.png'
plt.savefig(save_dir + filename, dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Removing rows with missing values
data_cleaned = data_normalized.dropna()

# Checking the shape of the cleaned dataset to confirm the rows were removed
data_cleaned.shape

In [ ]:
from sklearn.decomposition import PCA

# Assuming data is your DataFrame containing the data for PCA
data_for_pca = data_cleaned[['User_distance','User_X', 'User_Y', 'GazeOrigin_X', 'GazeOrigin_Y', 'Frechet_Distance', 'User_Roll', 'User_Pitch', 'User_Yaw', 'Behavior',  'AGV_distance', 'Trust']]

# Initialize PCA with the same number of components as before
pca = PCA(n_components=9)
pca_result = pca.fit_transform(data_for_pca)

# Get the explained variance ratio for each principal component
explained_variance_ratio = pca.explained_variance_ratio_

# Print the explained variance ratio for each principal component
for i, ratio in enumerate(explained_variance_ratio):
    print(f'Explained Variance Ratio for PC{i + 1}: {ratio:.2f}')

# Calculate the cumulative explained variance ratio
cumulative_variance_ratio = np.cumsum(explained_variance_ratio)

# Print the cumulative explained variance ratio
print('\nCumulative Explained Variance Ratio:')
for i, ratio in enumerate(cumulative_variance_ratio):
    print(f'PC{i + 1}: {ratio:.2f}')

# Determine which features to keep based on the cumulative explained variance ratio
num_components_to_keep = np.argmax(cumulative_variance_ratio >= 0.95) + 1
print(f'\nNumber of components to keep for 95% variance: {num_components_to_keep}')

# Determine which features contribute the most to each principal component
top_features_per_component = {}
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)

for pc, loading in enumerate(loadings.T):
    top_features = data_for_pca.columns[np.argsort(np.abs(loading))[::-1]][:3]  # Top 3 features per component
    top_features_per_component[f'PC{pc + 1}'] = top_features.tolist()

# Plot the explained variance and cumulative explained variance
plt.figure(figsize=(8, 6))
plt.plot(np.arange(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker='o', label='Explained Variance Ratio')
plt.plot(np.arange(1, len(cumulative_variance_ratio) + 1), cumulative_variance_ratio, marker='s', label='Cumulative Explained Variance Ratio')
plt.axhline(y=0.95, color='r', linestyle='--', label='95% Variance Threshold')

# Annotate the plot with the top contributing features for each principal component
for pc, top_features in top_features_per_component.items():
    annotation_text = f'{pc}: {", ".join(top_features)}'
    plt.annotate(annotation_text, xy=(int(pc[2:]), 0.95), xytext=(int(pc[2:]), 0.7 - int(pc[2:]) * 0.06),
                 arrowprops=dict(facecolor='black', arrowstyle='->'), fontsize=10, ha='center')

plt.xlabel('Number of Components')
plt.ylabel('Variance Ratio')
plt.title('Explained Variance and Cumulative Explained Variance')
plt.legend(loc='upper left', bbox_to_anchor=(1, 0.5))
plt.grid(True)

filename = 'Explained Variance and Cumulative Explained Variance.png'
plt.savefig(save_dir + filename, dpi=300, bbox_inches='tight')

plt.show()

In [ ]:
data = data.drop(columns=['Confidence', 'time_seconds'])

In [ ]:
# Setting up the figure
plt.figure(figsize=(12, 11))

# Check correlation matrix
correlation_matrix = data.corr()

#Draw the heat map
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', fmt=".2f", linewidths=0.5)

plt.title('Correlation Heatmap')

# Generate a timestamp
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

# Define the filename with timestamp
filename = f"Correlation_Heatmap_of_Trajectory_Data_{timestamp}.png"
'''
plt.savefig(save_dir + filename, dpi = 300)
'''
plt.show

## Importing Libraries

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [ ]:
# Checking for missing data in the selected features
missing_data = data.isnull().sum()

# Filtering out only those columns which have missing data
missing_data = missing_data[missing_data > 0]

missing_data

In [ ]:
# Removing rows with missing values
data_cleaned = data.dropna()

# Checking the shape of the cleaned dataset to confirm the rows were removed
data_cleaned.shape

In [ ]:
data = data_cleaned

In [ ]:
# Select features (excluding the 'Trust' column)
features = data.drop(columns=['Trust'])

# Select labels (using only the 'Trust' column)
labels = data['Trust']

print("Features shape:", features.shape)
print("Labels shape:", labels.shape)

In [ ]:
# Standardize the features
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Perform PCA
n_components = 40
pca = PCA(n_components=n_components)
features_pca = pca.fit_transform(features_scaled)

# Reconstruct the features from the principal components
features_reconstructed = pca.inverse_transform(features_pca)

# Rescale the reconstructed data back to the original scale
features_reconstructed_rescaled = scaler.inverse_transform(features_reconstructed)

In [ ]:
# Calculating the explained variance ratio
explained_variance_ratio = pca.explained_variance_ratio_
cumulative_variance = explained_variance_ratio.cumsum()

# Scree plot for the cumulative variance
plt.figure(figsize=(10,5))
plt.plot(features.columns, cumulative_variance, marker='o')  # Using column names as x-axis
plt.xlabel('Features')
plt.ylabel('Cumulative Explained Variance')
plt.title('Scree Plot')
plt.xticks(rotation=45, ha='right')  # Rotate labels for better readability
plt.grid(True)
plt.show()

In [ ]:
# Set the figure size
plt.figure(figsize=(12, 6))

# Creating the bar plot
plt.bar(features.columns[:n_components], explained_variance_ratio)

# Adding the labels and title
plt.xlabel('Feature')
plt.ylabel('Explained Variance Ratio')
plt.title('Variance Explained by Each Feature')

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, ha='right')

# Generate a timestamp
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

# Define the filename with timestamp
filename = f"Variability Explained by Each Feature_{timestamp}.png"
'''
plt.savefig(save_dir + filename, dpi = 300)
'''
# Display the plot
plt.show()

In [ ]:
# Extract all columns except 'trust' and 'confidence'
features = data.drop(columns=['Trust'])

# Standardize the features
features_standardized = (features - features.mean()) / features.std()

# Perform PCA
pca = PCA()
pca.fit(features_standardized)

# Eigenvalues and corresponding features
eigenvalues = pca.explained_variance_
feature_names = features.columns.tolist()

# Combine eigenvalues with corresponding feature names
eigenvalue_features = list(zip(eigenvalues, feature_names))

# Sort eigenvalue-feature pairs by eigenvalues in descending order
sorted_eigenvalue_features = sorted(eigenvalue_features, key=lambda x: x[0], reverse=True)

# Print eigenvalues along with corresponding features
print("Eigenvalues:")
for i, (eigenvalue, feature) in enumerate(sorted_eigenvalue_features):
    print(f"Eigenvalue {i+1}: {eigenvalue} - Feature: {feature}")

In [ ]:
# Set up the matplotlib figure
unique_agv_names = data['AGVname'].unique()
num_plots = len(unique_agv_names)
cols = 3  # Number of columns in the subplot grid
rows = -(-num_plots // cols)  # Calculate number of rows needed

sns.set_theme(style="white")
rs = np.random.RandomState(50)

# Set up the matplotlib figure
f, axes = plt.subplots(4, 4, figsize=(9, 9), sharex=True, sharey=True)

# Iterate over unique AGVnames and plot data
for ax, agv_name in zip(axes.flat, unique_agv_names):
    # Filter data for the current AGVname
    agv_data = data[data['AGVname'] == agv_name]
    # Create a cubehelix colormap to use with kdeplot
    cmap = sns.cubehelix_palette(start=s, light=1, as_cmap=True)
    # Generate and plot a random bivariate dataset
    x, y = rs.normal(size=(2, 50))
    sns.kdeplot(
        x=x, y=y,
        cmap=cmap, fill=True,
        clip=(-5, 5), cut=10,
        thresh=0, levels=15,
        ax=ax,
    )
    ax.set_axis_off()

ax.set(xlim=(-3.5, 3.5), ylim=(-3.5, 3.5))
f.subplots_adjust(0, 0, 1, 1, .08, .08)
    # Plot data for the current AGVname
    sns.scatterplot(data=agv_data, x='User_distance', y='Frechet_Distance', ax=ax)
    ax.set_title(agv_name)  # Set subplot title as AGVname
    ax.set_xlabel('User_distance')  # Set x-axis label
    ax.set_ylabel('Frechet_distance')  # Set y-axis label

# Adjust layout
plt.tight_layout()
plt.show()

#### Cross-Validation: By performing summary statistics for each column in the dataset, the objective was to eliminate columns where the summary statistics were essentially the same, indicating a low standard deviation (std). 
#### However, a low std on its own might not necessarily imply useless data; it should be compared with the mean. Consequently, I calculated the Coefficient of Variation for each column. 
#### If this value was less than 20%, the data in that column could be considered useless.

In [ ]:
# Display summary statistics
summary_statistics = data.describe()

# Print the summary statistics
print(summary_statistics)

In [ ]:
print(data.dtypes)

# Convert all columns to numeric
data = data.apply(pd.to_numeric, errors='coerce')

# Calculate CV for all columns
cv_values = (data.std() / data.mean()) * 100

# Display or print the calculated CV values
print("Coefficient of Variation for Each Column:")
print(cv_values)

In [ ]:
# Identify columns with absolute CV less than 20
columns_to_keep = cv_values[abs(cv_values) >= 20].index

# Ensure 'Behavior' column is included
if 'Behavior' not in columns_to_keep:
    columns_to_keep = columns_to_keep.insert(0, 'Behavior', 'AGVname')

# Create a new DataFrame with only the selected columns
filtered_data = data[columns_to_keep]

# Display the updated DataFrame
filtered_data.iloc[2000:2021]

# 6. Data Modeling

## Linear Regression for Trust with User_Distance and Frechet_Distance

In [ ]:
# Filter data for specific AGVname and PID
filtered_data = data[(data['AGVname'] == 13) & (data['PID'] == 4)]

# Filtered data for Behavior = 0 and Behavior = 1
behavior_0 = filtered_data[filtered_data['Behavior'] == 0]
behavior_1 = filtered_data[filtered_data['Behavior'] == 1]

# Scale the 'Frechet_Distance' and 'User_distance' variables
scaler = StandardScaler()
behavior_0_scaled = scaler.fit_transform(behavior_0[['Frechet_Distance', 'User_distance']])
behavior_1_scaled = scaler.transform(behavior_1[['Frechet_Distance', 'User_distance']])

# Update the scaled values in the original dataframes
behavior_0.loc[:, ['Frechet_Distance', 'User_distance']] = behavior_0_scaled
behavior_1.loc[:, ['Frechet_Distance', 'User_distance']] = behavior_1_scaled

# Concatenate the scaled data for both behaviors
concatenated_data = pd.concat([behavior_0, behavior_1])

# Melt the DataFrame to long-form
melted_data = pd.melt(concatenated_data, id_vars=['Behavior'], value_vars=['Trust', 'Frechet_Distance', 'User_distance'])

# Create catplot
plot = sns.catplot(data=melted_data, x='Behavior', y='value', hue='variable', kind='bar', errorbar='sd', palette="dark", alpha=.5, height=4)
plot.set(ylim=(0, 20))  
plt.title('Pair Box Plots for Trust, Frechet_Distance, and User_distance (Scaled)')
plt.show()

In [ ]:
selected_features = [
    'User_distance', 'Frechet_Distance', 'Trust','Behavior']

data_selected = data[selected_features]

# Display the head of the selected dataset
data_selected.head()

In [ ]:
grouped_data = data_selected.groupby('Behavior')

# Plotting individual heatmaps for each group
for group_name, group_df in grouped_data:
    # Extracting data for the current group
    x = group_df['User_distance']
    y = group_df['Frechet_Distance']
    z = group_df['Trust']

    # Create the hexbin plot for the heatmap
    plt.figure(figsize=(8, 6))
    plt.hexbin(x, y, C=z, gridsize=100, cmap='YlGnBu')  
    cb = plt.colorbar()
    cb.set_label('Trust')
    plt.xlabel('User_distance')
    plt.ylabel('Frechet Distance')
    plt.title(f'Heatmap of Trust vs. User_distance and Frechet Distance for Behavior: {group_name}')
    plt.show()

In [ ]:
# Slice the DataFrame to include only the first 200 data points
data_selected_200 = data_selected.iloc[1600:8000]

# Plot time series with error bands for each column
plt.figure(figsize=(12, 6))  # Adjust the figure size if needed

# Plot user_distance
sns.lineplot(data=data_selected_200['User_distance'], label='User Distance')
sns.lineplot(data=data_selected_200['Frechet_Distance'], label='Frechet Distance')
sns.lineplot(data=data_selected_200['Trust'], label='Trust')

plt.fill_between(range(len(data_selected_200)), data_selected_200['User_distance'] - data_selected_200['User_distance'].std(), data_selected_200['User_distance'] + data_selected_200['User_distance'].std(), alpha=0.3)
plt.fill_between(range(len(data_selected_200)), data_selected_200['Frechet_Distance'] - data_selected_200['Frechet_Distance'].std(), data_selected_200['Frechet_Distance'] + data_selected_200['Frechet_Distance'].std(), alpha=0.3)
plt.fill_between(range(len(data_selected_200)), data_selected_200['Trust'] - data_selected_200['Trust'].std(), data_selected_200['Trust'] + data_selected_200['Trust'].std(), alpha=0.3)

plt.xlabel('Index')  # Adjust x-axis label if needed
plt.ylabel('Values')  # Adjust y-axis label if needed
plt.title('Time Series with Error Bands (First 200 Data Points)')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 10))

# Plotting the relationship of each variable to Trust
for i, column in enumerate(data_selected.columns[:-1], 1):
    plt.subplot(2, 2, i)
    if data[column].dtype in ['int64', 'float64']:
        sns.scatterplot(x=data_selected[column], y=data_selected['Trust'])
    else:
        sns.boxplot(x=data_selected[column], y=data_selected['Trust'])
    plt.title(f'Relation of {column} to Trust')
    plt.xlabel(column)
    plt.ylabel('Trust')
    plt.tight_layout()

plt.show()

In [ ]:
# Splitting the dataset into training (80%) and testing (20%) sets
train_data, test_data = train_test_split(data_selected, test_size=0.2, random_state=42)

# Checking the shape of the training and testing sets
train_data.shape, test_data.shape

In [ ]:
# Separating the independent variables (X) and the target variable (y: Trust)
X_train = train_data.drop(columns=['Trust'])
y_train = train_data['Trust']

X_test = test_data.drop(columns=['Trust'])
y_test = test_data['Trust']

print('Shape of X_train: ', X_train.shape)
print('Shape of X_test: ', X_test.shape)
print('Shape of y_train: ', y_train.shape)
print('Shape of y_test: ', y_test.shape)

In [ ]:
# Preditors are scaled but the target (y) is not touched.
# We should scale both training and test partitions.
scaler = preprocessing.MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.fit_transform(X_test), columns=X_test.columns)

In [ ]:
X_train_scaled.head()

In [ ]:
X_test_scaled.head()

In [ ]:
y_train.head()

In [ ]:
# We use scatterplot matrix from seaborn (pairplot) to see the association between each pair of variables
sns.pairplot(data_selected)

# Generate a timestamp
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

# Define the filename using f-string or other appropriate method
filename = f"scatterplot_{timestamp}.png"

# Save the plot
plt.savefig(save_dir+filename, dpi=300)

plt.show() 

In [ ]:
# Assumption 3: correlations between the predictors are checked
# Assumptions 4 & 5 can be checked with statistical tests which are not in this course bondaries.
print(data_selected.drop(columns=['Trust']).corr())

In [ ]:
# Instantiate and fit the model

LM = LinearRegression()
LM.fit(X_train_scaled, y_train)

In [ ]:
# Predict the unseen observations

LM_preds = LM.predict(X_test_scaled)

In [ ]:
# Evaluate the training and test prediction errors
LM_MSE_test = mse(LM_preds, y_test)
LM_MSE_train = mse(LM.predict(X_train_scaled), y_train)

print('Linear regression test MSE: ', LM_MSE_test)
print('Linear regression training MSE: ', LM_MSE_train)

# Make more sense of the MSE (RRMSE = sqrt(MSE)/mean(y))

LM_RRMSE_test = np.sqrt(LM_MSE_test)/(y_test.mean())
LM_RRMSE_train = np.sqrt(LM_MSE_train)/(y_train.mean())

print('Linear regression test RRMSE: ', LM_RRMSE_test)
print('Linear regression training RRMSE: ', LM_RRMSE_train)

# Evaluate the training and test prediction errors using Mean Absolute Error (MAE)
LM_MAE_test = mae(LM_preds, y_test)
LM_MAE_train = mae(LM.predict(X_train_scaled), y_train)

print('Linear regression test MAE: ', LM_MAE_test)
print('Linear regression training MAE: ', LM_MAE_train)

# Calculate the Mean Absolute Percent Error (MAPE)
# Print the results
print('Linear regression test MAPE:', mape(y_test, LM_preds))

In [ ]:
# Create a DataFrame for the results
regression_results_df = pd.DataFrame({
    'Metric': ['MSE', 'MAE', 'RRMSE', 'MAPE'],
    'Training': [LM_MSE_train, LM_MAE_train, LM_RRMSE_train, 'N/A'],
    'Testing': [LM_MSE_test, LM_MAE_test, LM_RRMSE_test, mape(y_test, LM_preds)]
})

regression_results_df

In [ ]:
# Create a plot from the DataFrame
plt.figure(figsize=(8, 6))
plt.table(cellText=regression_results_df.values,
          colLabels=regression_results_df.columns,
          cellLoc = 'center', rowLoc = 'center',
          loc='center')
plt.axis('off')  # Hide the axes
plt.tight_layout()

# Generate a timestamp
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

# Define the filename using f-string or other appropriate method
filename = f"regression_metrics_{timestamp}.png"

# Save the plot
plt.savefig(save_dir+filename, dpi=300)

plt.show()

In [ ]:
# Linear model coefficients and intercept

print('coefficients:', LM.coef_)
print('intercept:', LM.intercept_)

## Random Forest Regressor for Trust with User_distance and Frechet_Distance

In [ ]:
RF = RandomForestRegressor(n_estimators = 100, random_state = 2023)
RF_param = {'max_depth': np.arange(8, 30)}
RF_GS = GridSearchCV(RF, RF_param, cv=5)
RF_GS.fit(X_train_scaled, y_train)
RF_GS.best_estimator_
# Get the best estimator
best_RF = RF_GS.best_estimator_

# Predict on the test set
y_pred = best_RF.predict(X_test_scaled)

In [ ]:
# Calculate MSE
MSE = mse(y_test, y_pred)

# Calculate MAE
MAE = mae(y_test, y_pred)

# Calculate RRMSE
RRMSE = np.sqrt(MSE) / np.mean(y_test)

# Calculate MAPE
MAPE = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print("Mean Squared Error (MSE):", MSE)
print("Mean Absolute Error (MAE):", MAE)
print("Relative Root Mean Squared Error (RRMSE):", RRMSE)
print("Mean Absolute Percentage Error (MAPE):", MAPE)

In [ ]:
# Format the metrics with 0.2f precision
MSE_formatted = "{:.2f}".format(MSE)
MAE_formatted = "{:.2f}".format(MAE)
RRMSE_formatted = "{:.2f}".format(RRMSE)
MAPE_formatted = "{:.2f}".format(MAPE)

# Create a table
rf_results_df = pd.DataFrame({
    'Metric': ['Mean Squared Error (MSE)', 'Mean Absolute Error (MAE)', 
               'Relative Root Mean Squared Error (RRMSE)', 'Mean Absolute Percentage Error (MAPE)'],
    'Value': [MSE_formatted, MAE_formatted, RRMSE_formatted, MAPE_formatted]
})

rf_results_df

In [ ]:
# Create a plot from the DataFrame
plt.figure(figsize=(8, 6))
plt.table(cellText=rf_results_df.values,
          colLabels=rf_results_df.columns,
          cellLoc = 'center', rowLoc = 'center',
          loc='center')
plt.axis('off')  # Hide the axes
plt.tight_layout()

# Generate a timestamp
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

# Define the filename using f-string or other appropriate method
filename = f"rf_metrics_{timestamp}.png"

# Save the plot
plt.savefig(save_dir+filename, dpi=300)

plt.show()

In [ ]:
train_accuracy = RF_GS.best_score_
test_accuracy = RF_GS.score(X_test_scaled, y_test)

print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import time

max_depths = np.arange(8, 30)
train_results = []
test_results = []

best_RF = RF_GS.best_estimator_
start = time.time()

for max_depth in max_depths:
    best_RF.set_params(max_depth=max_depth)
    best_RF.fit(X_train_scaled, y_train)
    train_accuracy = best_RF.score(X_train_scaled, y_train)
    train_results.append(train_accuracy)
    test_accuracy = best_RF.score(X_test_scaled, y_test)
    test_results.append(test_accuracy)
    
RFR_time = time.time() - start
print('Random Forrest Regressor time: ', RFR_time)

In [ ]:
plt.plot(max_depths, train_results, 'b', label='Train Accuracy')
plt.plot(max_depths, test_results, 'r', label='Test Accuracy')
plt.legend()
plt.ylabel('Accuracy')
plt.xlabel('max_depth')
plt.title('Effect of max_depth on Overfitting')

# Generate a timestamp
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

# Define the filename using f-string or other appropriate method
filename = f"Effect of max_depth on Overfitting_{timestamp}.png"

# Save the plot
plt.savefig(save_dir+filename, dpi=300)

plt.show()

In [ ]:
# Each random forest estimator is a combination of many decision trees

RF_GS.best_estimator_

In [ ]:
# Decision tree number 1

print(RF_GS.best_estimator_.estimators_[0])
print('Total number of base estimators: ', len(RF_GS.best_estimator_.estimators_))

## Other Regression Models

In [ ]:
from tabulate import tabulate

# define list of models (as tuples)
random = 2023
Regressor_models = [ 
          ('KNN', KNeighborsRegressor(n_neighbors=5)),
          ('RegressionTree', DecisionTreeRegressor(max_depth=4, random_state=random))
        ]

# Define lists to store results
Regressor_results = []
Regressor_names = []

# Define the header separately
header = ["Model", "Mean Absolute Percent Error", "Mean Squared Error", "Root Mean Squared Error", "Relative RMSE", "Mean Absolute Error"]

start = time.time()
# Train and evaluate each regression model
for name, model in Regressor_models:
    # Train the regression model
    fitted = model.fit(X_train_scaled, y_train)
    
    # Make predictions
    pred = fitted.predict(X_test_scaled)
    
    # Calculate Mean Absolute Percent Error (MAPE)
    current_mape = mape(y_test, pred)
    
    # Calculate Mean Squared Error (MSE)
    current_mse = mse(y_test, pred)
    
    # Calculate Root Mean Squared Error (RMSE)
    current_rmse = np.sqrt(current_mse)
    
    # Calculate Relative Root Mean Squared Error (RRMSE)
    rrmse_denominator = np.mean(y_test)
    current_rrmse = current_rmse / rrmse_denominator
    
    # Calculate Mean Absolute Error (MAE)
    current_mae = mae(y_test, pred)
    
    # Print the results
    print(tabulate([[name, f'{current_mape:.2f}', f'{current_mse:.2f}', f'{current_rmse:.2f}', f'{current_rrmse:.2f}', f'{current_mae:.2f}']],header, tablefmt="fancy_grid"))
    
    # Append results to lists
    Regressor_results.append({
        'Model': name,
        'MAPE': current_mape,
        'MSE': current_mse,
        'RMSE': current_rmse,
        'RRMSE': current_rrmse,
        'MAE': current_mae
    })
    Regressor_names.append(name)
R_time = time.time() - start
print('Other Regressors time: ', R_time)

In [ ]:
# We can see the effect of max_depth on overfitting

maxdepth = np.arange(1,15)
train_results = []
test_results = []
start = time.time()
for m in maxdepth:
    model = DecisionTreeRegressor(max_depth=m, random_state=2020)
    model.fit(X_train_scaled, y_train)
    train_pred = model.predict(X_train_scaled)
    train_mse = mse(train_pred, y_train)
    train_results.append(train_mse)
    test_pred = model.predict(X_test_scaled)
    test_mse = mse(test_pred, y_test)
    test_results.append(test_mse)
DT_time = time.time() - start
print('Other Regressors time: ', DT_time)

In [ ]:
plt.plot(maxdepth, train_results, 'b', label='Train MSE')
plt.plot(maxdepth, test_results, 'r', label='Test MSE')
plt.legend()
plt.ylabel('MSE')
plt.xlabel('max_depth')

# Generate a timestamp
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

# Define the filename using f-string or other appropriate method
filename = f"Decision Tree Regressor_{timestamp}.png"

# Save the plot
plt.savefig(save_dir+filename, dpi=300)

plt.show()

## Building Regression 

In [ ]:
from sklearn.linear_model import LinearRegression

In [ ]:
# Error evaluation libraries
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import mean_absolute_percentage_error as mape
from sklearn.metrics import classification_report

In [ ]:
# Calculating the total change in x and y, and count for each unique AGV_number, Behavior, and PID
result = (trajectory_deviation_data_set.groupby(['AGV_number', 'SCN', 'PID'])
           .apply(lambda group: pd.Series({
               'Total_Change_X': group.iloc[-1]['User_X'] - group.iloc[0]['User_X'],
               'Total_Change_Y': group.iloc[-1]['User_Y'] - group.iloc[0]['User_Y'],
               'Count': group.shape[0],
           }))
           .reset_index())

In [ ]:
# Calculating incremental change in x and y
result['Incremental_X'] = result['Total_Change_X'] / result['Count']
result['Incremental_Y'] = result['Total_Change_Y'] / result['Count']

In [ ]:
result

In [ ]:
# Set the style for the plot
sns.set(style="whitegrid")

# Create a scatter plot for Incremental_X and Incremental_Y columns
sns.scatterplot(x='Incremental_X', y='Incremental_Y', data=result)

# Add labels and title
plt.xlabel('Incremental_X')
plt.ylabel('Incremental_Y')

plt.title('Scatter Plot of Incremental_X vs Incremental_Y')

# Show the plot
plt.show()

In [ ]:
# Sort the DataFrame based on the 'PID' column
result_sorted = result.sort_values(by='PID')

# Display the sorted DataFrame
result_sorted

In [ ]:
# Creating the Expected_X and Expected_Y columns in the original DataFrame
trajectory_deviation_data_set['Expected_X'] = trajectory_deviation_data_set.apply(
    lambda row: row['User_X'] + result.loc[
        (result['AGV_number'] == row['AGV_number']) & (result['SCN'] == row['SCN']),
        'Incremental_X'
    ].values[0],
    axis=1
)


trajectory_deviation_data_set['Expected_Y'] = trajectory_deviation_data_set.apply(
    lambda row: row['User_Y'] + result.loc[
        (result['AGV_number'] == row['AGV_number']) & (result['SCN'] == row['SCN']),
        'Incremental_Y'
    ].values[0],
    axis=1
)


**The Fréchet distance between two points is a measure of similarity between two curves or two sets of points in a metric space. For two points in a two-dimensional space, the Fréchet distance is simply the Euclidean distance between the points.**

In [ ]:
trajectory_deviation_data_set['D_from_T'] = np.sqrt(
    ((trajectory_deviation_data_set['User_X'] - trajectory_deviation_data_set['Expected_X']) ** 2) +
    ((trajectory_deviation_data_set['User_Y'] - trajectory_deviation_data_set['Expected_Y']) ** 2)
) 
trajectory_deviation_data_set.head()

In [ ]:
#Checking the change
trajectory_deviation_data_set.head()

In [ ]:
# Divide AGV_distance by 100
trajectory_deviation_data_set['AGV_distance'] = trajectory_deviation_data_set['AGV_distance'] / 100

sns.set_theme(style="white")

# Create JointGrid and plot KDE with marginal histograms
g = sns.JointGrid(data=trajectory_deviation_data_set, x="AGV_distance", y="D_from_T", space=0)

g.plot_joint(sns.kdeplot, fill=True,
             thresh=0, cmap="rocket", levels=10) 

g.plot_marginals(sns.histplot, color="#03051A", alpha = 1, bins=40)


plt.tight_layout()

# Define the filename using f-string
filename = 'Fréchet Distance for every two points'

plt.savefig(save_dir + filename, dpi = 300)

# Show the plot
plt.show()

In [ ]:
# Set the Seaborn style
sns.set_theme(style="ticks")

# Load the planets dataset and initialize the figure
g = sns.JointGrid(data=trajectory_deviation_data_set, x="AGV_distance", y="D_from_T", space=0)

# Set a log scaling on the y-axis
g.ax_joint.set(yscale="log")

# Create an inset legend for the histogram colorbar
cax = g.figure.add_axes([0, .55, .02, .2])

# Add the joint and marginal histogram plots
g.plot_joint(
    sns.histplot, discrete=(True, False),
    cmap="rocket", pmax=.8, cbar=True, cbar_ax=cax
)
g.plot_marginals(sns.histplot, element="step", color="#03012d")

plt.tight_layout()

# Define the filename using f-string
filename = 'Fréchet Distance for every two points - 2'

plt.savefig(save_dir + filename, dpi = 300)

# Show the plot
plt.show()

In [ ]:
# Plotting the bar plot
plt.figure(figsize=(10, 6))
sns.set(style="whitegrid")

for scn_value in [0, 1]:
    scn_data = frechet_distance_of_trajectories[frechet_distance_of_trajectories['SCN'] == scn_value]
    color = 'red' if scn_value == 0 else 'green'
    sns.barplot(x='AGV_number', y='Frechet_Distance', data=scn_data, color=color, label=f'SCN {scn_value}')

plt.title('Frechet Distance for Each AGV_number')
plt.xlabel('AGV_number')
plt.ylabel('Frechet Distance')
plt.legend()
plt.show()

# Plotting the scatter plot
plt.figure(figsize=(10, 6))

for scn_value in [0, 1]:
    scn_data = trajectory_deviation_data_set[trajectory_deviation_data_set['SCN'] == scn_value]
    color = 'red' if scn_value == 0 else 'green'
    sns.scatterplot(x='AGV_X', y='AGV_Y', data=scn_data, color=color, label=f'SCN {scn_value}')

plt.title('Scatter Plot of AGV_X and AGV_Y for Each AGV_number')
plt.xlabel('AGV_X')
plt.ylabel('AGV_Y')
plt.legend()
plt.show()

#'''
### Changed 3/6/24 by Thomas Lenz
for scn_value in [0, 1]:
    scn_data = trajectory_deviation_data_set[trajectory_deviation_data_set['SCN'] == scn_value]
    color = 'red' if scn_value == 0 else 'green'
    sns.scatterplot(x='AGV_X', y='AGV_Y', data=scn_data, color=color, label=f'SCN {scn_value}')
    sns.scatterplot(x='User_X', y='User_Y', data=scn_data, color=color, label='User')
    plt.title('Scatter Plot of AGV_X and AGV_Y for Each AGV_number')
    plt.xlabel('AGV_X')
    plt.ylabel('AGV_Y')
    plt.legend()
    plt.show()
    
'''
for run in trajectory_deviation_data_set:
    color = 'red' if run['SCN'] == '0' else 'green'
    sns.scatterplot(x='AGV_X', y='AGV_Y', data=run, color=color, label=f'SCN {scn_value}')
    sns.scatterplot(x='User_X', y='User_Y', data=run, color=color, label='User')
'''
#'''



In [ ]:
# Preprocess data to get counts for each combination
count_data = frechet_distance_of_trajectories.groupby(['PID', 'AGV_number', 'SCN']).size().reset_index(name='Count')

# Merge counts with the Frechet distance data
merged_data = pd.merge(frechet_distance_of_trajectories, count_data, on=['PID', 'AGV_number', 'SCN'])

# Create the displot
sns.set_theme(style="darkgrid")
g = sns.displot(
    merged_data, x="Frechet_Distance", col="AGV_number", row="SCN",
    binwidth=3, height=3, facet_kws=dict(margin_titles=True),
    hue="AGV_number", multiple="stack", shrink=0.8,
)
g.set_axis_labels("Frechet Distance", "Count")
g.set_titles(col_template="{col_name}", row_template="{row_name}")
plt.show()

# 4. Data Modeling

## 4.1 Regression for User_Distance

#### Now we want to perform regression for user_distance without the behavior column. 
#### Our idea is to predict how far a user would go in the next second. 
#### In this case, deleting the behavior is essential, as having that would hinder the interpretation of the distance. For example, if it's moving, we're sure that the user would go at least 1 cm far.

#### We make a copy of the data, ensuring it's not going to be altered, as we are about to drop an important column named "behavior."

In [ ]:
data_for_user_distance = data_with_threshold_54.copy()

data_for_user_distance = data_for_user_distance.drop(columns=['Behavior'])

data_for_user_distance.head()

In [ ]:
# Assuming data_with_threshold_54 is your DataFrame
# Select columns with boolean data points
bool_columns = data_for_user_distance.select_dtypes(include='bool').columns

# Drop columns with boolean data points
data_for_user_distance_w_bool = data_for_user_distance.drop(columns=bool_columns)

### Import needed libraries

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

In [ ]:
# Error evaluation libraries
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import mean_absolute_percentage_error as mape
from sklearn.metrics import classification_report

In [ ]:
# Splitting the dataset into training (80%) and testing (20%) sets
train_data_for_user, test_data_for_user = train_test_split(data_for_user_distance_w_bool, test_size=0.1, random_state=42) #For_user" here means "for user distance," which refers to predicting user distance via regression.

# Checking the shape of the training and testing sets
train_data_for_user.shape, test_data_for_user.shape

In [ ]:
# Separating the independent variables (X) and the target variable (y: SalePrice)
X_train_for_user = train_data_for_user.drop(columns=['User_distance'])
y_train_for_user = train_data_for_user['User_distance']

X_test_for_user = test_data_for_user.drop(columns=['User_distance'])
y_test_for_user = test_data_for_user['User_distance']

print('Shape of X_train: ', X_train_for_user.shape)
print('Shape of X_test: ', X_test_for_user.shape)
print('Shape of y_train: ', y_train_for_user.shape)
print('Shape of y_test: ', y_test_for_user.shape)

### Scaling the data

In [ ]:
# Preditors are scaled but the target (y) is not touched.
# We should scale both training and test partitions.

scaler = preprocessing.MinMaxScaler()
X_train_for_user_scaled = pd.DataFrame(scaler.fit_transform(X_train_for_user), columns=X_train_for_user.columns)
X_test_for_user_scaled = pd.DataFrame(scaler.fit_transform(X_test_for_user), columns=X_test_for_user.columns)

In [ ]:
X_train_for_user_scaled.head()

In [ ]:
X_test_for_user_scaled.head()

In [ ]:
# Instantiate and fit the model
LM = LinearRegression()
LM.fit(X_train_for_user, y_train_for_user)

In [ ]:
# Predict the unseen observations
LM_preds = LM.predict(X_test_for_user)

In [ ]:
# Linear model coefficients and intercept
print('coefficients:', LM.coef_)
print('intercept:', LM.intercept_)

### Mean Squared Error (MSE)

In [ ]:
# Evaluate the training and test prediction errors
LM_MSE_test = mse(LM_preds,y_test_for_user)
LM_MSE_train = mse(LM.predict(X_train_for_user), y_train_for_user)

print('Linear regression test MSE: ', LM_MSE_test)
print('Linear regression training MSE: ', LM_MSE_train)

### Root Mean Squared Error (RMSE)

In [ ]:
# Calculate the RMSE for the test set
LM_RMSE_test = np.sqrt(LM_MSE_test)
# Calculate the RMSE for the training set
LM_RMSE_train = np.sqrt(LM_MSE_train)

# Print the results
print('Linear regression test RMSE: {:.2f}'.format(LM_RMSE_test))
print('Linear regression training RMSE: {:.2f}'.format(LM_RMSE_train))

### Relative Root Mean Squared Error (RRMSE)

In [ ]:
# Make more sense of the MSE (RRMSE = sqrt(MSE)/mean(y))
LM_RRMSE_test = np.sqrt(LM_MSE_test)/(y_test_for_user.mean())
LM_RRMSE_train = np.sqrt(LM_MSE_train)/(y_train_for_user.mean())

print('Linear regression test RRMSE: ', LM_RRMSE_test)
print('Linear regression training RRMSE: ', LM_RRMSE_train)

### Mean Absolute Error (MAE)

In [ ]:
# Evaluate the training and test prediction errors using Mean Absolute Error (MAE)
LM_MAE_test = mae(LM_preds, y_test_for_user)
LM_MAE_train = mae(LM.predict(X_train_for_user), y_train_for_user)

print('Linear regression test MAE: ', LM_MAE_test)
print('Linear regression training MAE: ', LM_MAE_train)

### Mean Absolute Percent Error (MAPE)

In [ ]:
# Calculate the Mean Absolute Percent Error (MAPE)
# Print the results
print(mape(y_test_for_user, LM_preds))

In [ ]:
from tabulate import tabulate

# define list of models (as tuples)
random = 2023
Regressor_models = [
          ('LinearRegression', LinearRegression()), 
          ('KNN', KNeighborsRegressor(n_neighbors=10)),
          ('RegressionTree', DecisionTreeRegressor(max_depth=4, random_state=random)),
          ('Bagging', BaggingRegressor(n_estimators=100, random_state=random)), 
          ('RandomForest', RandomForestRegressor(n_estimators=100, max_depth=6, random_state=random)),
          ('GradientBoosting', GradientBoostingRegressor(n_estimators=100, max_depth=2, 
                                                         learning_rate=0.1, random_state=random)),
          ('XGBoost', XGBRegressor(n_estimators=100, max_depth=2, learning_rate=0.1, random_state=random)),
        ]

# Define lists to store results
Regressor_results = []
Regressor_names = []

# Define the header separately
header = ["Model", "Mean Absolute Percent Error", "Mean Squared Error", "Root Mean Squared Error", "Relative RMSE", "Mean Absolute Error"]

# Train and evaluate each regression model
for name, model in Regressor_models:
    # Train the regression model
    fitted = model.fit(X_train_for_user, y_train_for_user)
    
    # Make predictions
    pred = fitted.predict(X_test_for_user)
    
    # Calculate Mean Absolute Percent Error (MAPE)
    current_mape = mape(y_test_for_user, pred)
    
    # Calculate Mean Squared Error (MSE)
    current_mse = mse(y_test_for_user, pred)
    
    # Calculate Root Mean Squared Error (RMSE)
    current_rmse = np.sqrt(current_mse)
    
    # Calculate Relative Root Mean Squared Error (RRMSE)
    rrmse_denominator = np.mean(y_test_for_user)
    current_rrmse = current_rmse / rrmse_denominator
    
    # Calculate Mean Absolute Error (MAE)
    current_mae = mae(y_test_for_user, pred)
    
    # Print the results
    print(tabulate([[name, f'{current_mape:.2f}', f'{current_mse:.2f}', f'{current_rmse:.2f}', f'{current_rrmse:.2f}', f'{current_mae:.2f}']],header, tablefmt="fancy_grid"))
    
    # Append results to lists
    Regressor_results.append({
        'Model': name,
        'MAPE': current_mape,
        'MSE': current_mse,
        'RMSE': current_rmse,
        'RRMSE': current_rrmse,
        'MAE': current_mae
    })
    Regressor_names.append(name)

### Quantile Regression 

In [ ]:
from sklearn.utils.fixes import parse_version, sp_version

# This is line is to avoid incompatibility if older SciPy version.
# You should use `solver="highs"` with recent version of SciPy.
solver = "highs" if sp_version >= parse_version("1.6.0") else "interior-point"

In [ ]:
from sklearn.linear_model import QuantileRegressor
import numpy as np

# Assuming you have your own dataset: X_train, X_test, y_train, y_test

quantiles = [0.05, 0.5, 0.95]
predictions = {}
out_bounds_predictions = np.zeros_like(y_test_for_user, dtype=np.bool_)

for quantile in quantiles:
    qr = QuantileRegressor(quantile=quantile, alpha=0, solver=solver)  # You can adjust the solver as needed
    y_pred_qr = qr.fit(X_train_for_user, y_train_for_user).predict(X_test_for_user)
    predictions[quantile] = y_pred_qr

    if quantile == min(quantiles):
        out_bounds_predictions = np.logical_or(out_bounds_predictions, y_pred_qr >= y_test_for_user)
    elif quantile == max(quantiles):
        out_bounds_predictions = np.logical_or(out_bounds_predictions, y_pred_qr <= y_test_for_user)


In [ ]:
import matplotlib.pyplot as plt

# Plot the true mean
plt.plot(X_test_for_user['AGV_distance'], y_test_for_user, color="black", linestyle="dashed", label="True mean")

# Plot quantile predictions
for quantile, y_pred_qr in predictions.items():
    plt.plot(X_test_for_user['AGV_distance'], y_pred_qr, label=f"Quantile: {quantile}")

# Scatter points outside the interval
plt.scatter(
    X_test_for_user['AGV_distance'][out_bounds_predictions],
    y_test_for_user[out_bounds_predictions],
    color="black",
    marker="+",
    alpha=0.5,
    label="Outside interval",
)

# Scatter points inside the interval
plt.scatter(
    X_test_for_user['AGV_distance'][~out_bounds_predictions],
    y_test_for_user[~out_bounds_predictions],
    color="black",
    alpha=0.5,
    label="Inside interval",
)

plt.legend()
plt.xlabel("AGV_distance")
plt.ylabel("y_test_for_user")
plt.title("Quantiles of heteroscedastic Normal distributed target")
plt.show()

### Comparing QuantileRegressor and LinearRegression

In [ ]:
print(f"""Testing error
    {LinearRegression.__class__.__name__}:
    MAE = {mae(y_test_for_user, LM_preds):.3f}
    MSE = {mse(y_test_for_user, LM_preds):.3f}
    {QuantileRegressor.__class__.__name__}:
    MAE = {mae(y_test_for_user, y_pred_qr):.3f}
    MSE = {mse(y_test_for_user, y_pred_qr):.3f}
    """)

### Zero-Inflated Poisson Regression

#### We want to know whether the user_distance distribution is skewed over zero since now we have a threshold of 5.4, which below this number we assume the user made 0 cm distance. This is important because we want to make sure our assumption regarding distance movements following a Poisson distribution is correct.

In [ ]:
# Assuming data_with_threshold_54 is your DataFrame
data_for_user_distance_w_bool['User_distance'] = data_for_user_distance_w_bool['User_distance'].apply(lambda x: 0 if x < 5.4 else x)

data_for_user_distance_w_bool

In [ ]:
# Replace 'User_distance' with the actual column name
data_for_user_distance_w_bool['User_distance'] = np.ceil(data_for_user_distance_w_bool['User_distance']).astype(int)

# Display the modified DataFrame
print(data_for_user_distance_w_bool)

In [ ]:
# Assuming data_for_user_distance_w_bool is your DataFrame
data_for_user_distance_w_bool.to_csv('data_for_user_distance_w_bool.csv', index=False)

In [ ]:
sns.distplot(data_for_user_distance_w_bool['User_distance'], kde=False, hist_kws={'edgecolor': 'black'})

# Save the figure as an image
filename = 'Zero-Inflated_test_result.png'
plt.savefig(filename, format='png', bbox_inches='tight')

plt.show()

In [ ]:
data_for_user_distance_w_bool.columns

In [ ]:
data_for_user_distance_w_bool['User_distance'].unique

In [ ]:
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklego.meta import ZeroInflatedRegressor

In [ ]:
zir = ZeroInflatedRegressor(
    classifier=RandomForestClassifier(random_state=0),
    regressor=RandomForestRegressor(random_state=0)
)

lr = Ridge(random_state=0)

print('ZIR (RFC+RFR) R²:', cross_val_score(zir, X_train_for_user_scaled, y_train_for_user).mean())
print('RFR R²:', cross_val_score(RandomForestRegressor(random_state=0), X_train_for_user_scaled, y_train_for_user).mean())

In [ ]:
# Print the table-like structure
print("Results:")
print("{:<30} {:<20}".format("ZIR (RFC+RFR) R²", cross_val_score(zir, X_train_for_user_scaled, y_train_for_user).mean()))
print("{:<30} {:<20}".format("RFR R²", cross_val_score(RandomForestRegressor(random_state=0), X_train_for_user_scaled, y_train_for_user).mean()))

In [ ]:
data_for_user_distance_w_bool['User_distance'].isna().values.any()

In [ ]:
print(y_train_for_user.dtypes)
print(X_train_for_user.dtypes)
print(X_train_for_user.dtypes)

In [ ]:
# Convert to numpy arrays
y_train_array = np.asarray(y_train_for_user)
X_train_array = np.asarray(X_train_for_user)
X_infl_array = np.asarray(X_train_for_user)

In [ ]:
import statsmodels.api as sm

# Fit the model
model = sm.ZeroInflatedPoisson(endog=y_train_array, exog=X_train_array, exog_infl=X_infl_array, inflation='logit')
mdf = model.fit()

In [ ]:
# Fit the model to the data
zi_regressor.fit(X_train_for_user_scaled, y_train_for_user)

# Make predictions on new data
y_pred_zir = zi_regressor.predict(X_test_for_user_scaled)

In [ ]:
# Python program to illustrate
# how to estimate quantile regression 
import statsmodels.api as sm
import statsmodels.formula.api as smf

np.random.seed(0)

# Number of rows
rows = 20

# Constructing Distance column
Distance = np.random.uniform(1, 10, rows)

# Constructing Emission column
Emission = 40 + Distance + np.random.normal(loc=0,
											scale=.25*Distance,
											size=20)

# Creating the data set
df = pd.DataFrame({'Distance': Distance, 
				'Emission': Emission})

# fit the model
model = smf.quantreg('Emission ~ Distance',
					df).fit(q=0.7)

# view model summary
print(model.summary())


In [ ]:
from sklearn.model_selection import KFold
from skgarden import RandomForestQuantileRegressor

# Create a quantile regression forest
qr = QuantileRegressor()

# Fit the quantile regression forest to the training data
qr.fit(X_train_for_user, y_train_for_user)

# Make predictions on the test data
y_pred = qr.predict(X_test_for_user)

# Print the predictions
print(y_pred)

## 4.2. Behavior prediction
#### Classification models for this processed dataset: 
#### we aim to classify behaviors based on other attributes. 
#### Essentially, we want to determine, given all the other independent variables, whether we can predict the user's decision to stop or continue moving.

In [ ]:
data_with_Behavior = data_with_distance_difference_1.drop(columns=['User_distance'])

In [ ]:
# One-hot encoding for categorical feature decision
# The data will be read in with data types. But you can specify the data types when read in csv files.

dummies = pd.get_dummies(data_with_Behavior.drop(columns=['Behavior']))
dummies['class'] = data_with_Behavior['Behavior']
X_decision = dummies.drop(columns='class')
y_decision = dummies['class']
y_decision = y_decision.replace('Stop', 0)
y_decision = y_decision.replace('Moving', 1)

### Splitting the dataset

In [ ]:
# Splitting the dataset into training (80%) and testing (20%) sets
X_decision.columns = X_decision.columns.astype(str)
X_decision_train, X_decision_test, y_decision_train, y_decision_test = train_test_split(X_decision, y_decision, test_size=0.2, random_state=2023)

In [ ]:
# Checking the shape of the training and testing sets
X_decision_train.shape, X_decision_test.shape, y_decision_train.shape, y_decision_test.shape

### 1.3.2. Decision Tree Classifier for Behavior

#### Import Libraries

In [ ]:
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

# data and preprocessing libraries
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn import preprocessing

In [ ]:
# Hyperparameter tuning and model fitting
DT = DecisionTreeClassifier(random_state=2023)
DT_param = {'max_depth': np.arange(1, 15)}
DT_GS = GridSearchCV(DT, DT_param, cv=5)
DT_GS.fit(X_decision_train, y_decision_train)
DT_GS.best_estimator_

In [ ]:
# Prediction and model evaluation
DT_preds = DT_GS.predict(X_decision_test)
accuracy_dtc = accuracy_score(DT_preds, y_decision_test)
print("Test accuracy:", accuracy_dtc)
print("Training accuracy:", accuracy_score(DT_GS.predict(X_decision_train), y_decision_train))

In [ ]:
# We can see the effect of max_depth on overfitting
maxdepth = np.arange(1,15)
train_results = []
test_results = []

for m in maxdepth:
    model = DecisionTreeClassifier(max_depth=m, random_state=2023)
    model.fit(X_decision_train, y_decision_train)
    train_pred = model.predict(X_decision_train)
    train_accuracy = accuracy_score(train_pred, y_decision_train)
    train_results.append(train_accuracy)
    test_pred = model.predict(X_decision_test)
    test_accuracy = accuracy_score(test_pred, y_decision_test)
    test_results.append(test_accuracy)

plt.plot(maxdepth, train_results, 'b', label='Train Accuracy')
plt.plot(maxdepth, test_results, 'r', label='Test Accuracy')
plt.legend()
plt.ylabel('Accuracy')
plt.xlabel('max_depth')
plt.show()

In [ ]:
# Visualizing tree

fig = plt.figure(figsize=(20,10))
_ = tree.plot_tree(DT_GS.best_estimator_, filled=True, feature_names=list(X_decision.columns), fontsize=15)

### Splitting the dataset

In [ ]:
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import mean_squared_error
from sklearn.metrics import accuracy_score

In [ ]:
# Splitting the dataset into training (80%) and testing (20%) sets
train_data, test_data = train_test_split(data_with_distance_difference_1, test_size=0.2, random_state=42)

# Checking the shape of the training and testing sets
train_data.shape, test_data.shape

In [ ]:
# Separating the independent variables (X) and the target variable (y: SalePrice)
X_train = train_data.drop(columns=['Behavior'])
y_train = train_data['Behavior']

X_test = test_data.drop(columns=['Behavior'])
y_test = test_data['Behavior']

print('Shape of X_Train: ', X_train.shape)
print('Shape of X_Test: ', X_test.shape)
print('Shape of y_Train: ', y_train.shape)
print('Shape of y_Test: ', y_test.shape)

In [ ]:
# define list of models (as tuples)
random = 2023
Classifier_models = [
    ('LogReg', LogisticRegression()),
    ('KNN', KNeighborsClassifier(n_neighbors=10)),
    ('RegressionTree', DecisionTreeClassifier(max_depth=4, random_state=random)),
    ('Bagging', BaggingClassifier(n_estimators=100, random_state=random)),
    ('RandomForest', RandomForestClassifier(n_estimators=100, max_depth=6, random_state=random)),
    ('GradientBoosting', GradientBoostingClassifier(n_estimators=100, max_depth=2,
                                                     learning_rate=0.1, random_state=random)),
    ('XGBoost', XGBClassifier(n_estimators=100, max_depth=2, learning_rate=0.1, random_state=random)),
    ('LightGBM', LGBMClassifier(n_estimators=100, max_depth=4, learning_rate=0.2, random_state=random)),
]

In [ ]:
Classifier_results = []
Classifier_names = []

for name, model in Classifier_models:
    fitted = model.fit(X_train, y_train)
    pred = fitted.predict(X_test)
    accuracy = accuracy_score(y_test, pred)
    #print(name, 'accuracy: %.2f' % accuracy)
    Classifier_results.append(accuracy)
    Classifier_names.append(name)

In [ ]:
sns.barplot(x=Classifier_results, y=Classifier_names)
plt.xlabel('Test Accuracy')
plt.show()

#### This is the relative distance of the AGV from the user scatter plot. 
#### Each plot represents a specific AGV and its behavior towards our 13 users.

In [ ]:
'''
# Get unique PID values
unique_pids = data_with_distance_vr_2['PID'].unique()

# Create a color map for PID values
color_map = plt.cm.get_cmap('tab10', len(unique_pids))  # You can choose a different colormap if needed

# Group by 'AGV_number' and 'SCN'
grouped_data = data_with_distance_vr_2.groupby(['AGV_number', 'SCN'])

# Create scatter plots for each group
for group, group_data in grouped_data:
    plt.figure() 
    
    for pid, pid_data in group_data.groupby('PID'):
        pid_color = color_map(unique_pids.tolist().index(pid))  # Get color for each PID
        plt.scatter(pid_data.index, pid_data['AGV_distance'], c=np.array([pid_color]), label=f'PID {pid}')
    
    # Set labels and title
    plt.xlabel('Index')
    plt.ylabel('AGV_distance')
    plt.title(f'Scatter Plot of AGV_distance for {group}')
    
    # Show the plot
    plt.legend()
    plt.show()
'''